# 07. Semantic Search with Sentence Embeddings


## 1. What you will build

- A semantic search system using Sentence Transformers + FAISS.
- A keyword baseline using TF-IDF for comparison.
- Retrieval quality evaluation with Recall@k and MRR@k.


## 2. When to use this in real companies

Use this approach when users search with varied wording or paraphrases and exact keyword matching misses relevant results. Common scenarios are support portals, internal documentation search, and knowledge retrieval assistants.


## 3. Business goal

Improve document retrieval relevance for user queries by comparing semantic and lexical retrieval methods.


## 4. Imports and setup


In [9]:
from pathlib import Path

import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import logging
import warnings
import io
from contextlib import redirect_stdout
from huggingface_hub.utils import logging as hf_logging
from transformers.utils import logging as tr_logging

# Reduce noisy HF/Sentence-Transformers startup warnings in notebooks.
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
hf_logging.set_verbosity_error()
tr_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*You are sending unauthenticated requests to the HF Hub.*")
warnings.filterwarnings("ignore", message=".*BertModel LOAD REPORT.*")
warnings.filterwarnings("ignore", message=".*UNEXPECTED.*")


## 5. Load datasets from `data/07_data`


In [10]:
CORPUS_PATH = Path("../data/07_data/knowledge_base.csv")
QUERIES_PATH = Path("../data/07_data/search_queries.csv")

corpus = pd.read_csv(CORPUS_PATH)
queries = pd.read_csv(QUERIES_PATH)

print(f"Corpus size: {len(corpus)}")
print(f"Queries: {len(queries)}")
corpus.head()

Corpus size: 15
Queries: 15


,doc_id,text
0,faq_01,Reset your password from account settings usin...
1,faq_02,Invoices are generated on the first day of eac...
2,faq_03,Shipping address can be updated before dispatch.
3,faq_04,Enable two-factor authentication in security s...
4,faq_05,Refund reviews are completed in five business ...


In [11]:
queries.head()

,query,expected_doc_id
0,How do I recover my account password?,faq_01
1,When are monthly bills created?,faq_02
2,Can I change delivery address after ordering?,faq_03
3,Where can I activate MFA?,faq_04
4,How long do refund checks take?,faq_05


## 6. Semantic retrieval (SentenceTransformer + FAISS)


In [12]:
with redirect_stdout(io.StringIO()):
    model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(corpus["text"].tolist(), normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype("float32"))


def semantic_search(query: str, k: int = 3):
    """Return top-k semantic matches for one query."""
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_emb, k)
    out = corpus.iloc[idx[0]].copy()
    out["score"] = scores[0]
    return out[["doc_id", "text", "score"]].reset_index(drop=True)

semantic_search("How can I reset my account password?")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,doc_id,text,score
0,faq_01,Reset your password from account settings usin...,0.746654
1,faq_04,Enable two-factor authentication in security s...,0.325375
2,faq_07,Single sign-on is available on enterprise subs...,0.288227


## 7. Keyword baseline (TF-IDF cosine)


In [13]:
tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(corpus["text"])


def keyword_search(query: str, k: int = 3):
    """Return top-k keyword matches for one query."""
    q_vec = tfidf.transform([query])
    scores = cosine_similarity(q_vec, X)[0]
    idx = scores.argsort()[-k:][::-1]
    out = corpus.iloc[idx].copy()
    out["score"] = scores[idx]
    return out[["doc_id", "text", "score"]].reset_index(drop=True)

keyword_search("How can I reset my account password?")

,doc_id,text,score
0,faq_01,Reset your password from account settings usin...,0.792424
1,faq_14,SSO just-in-time provisioning is supported for...,0.000000
2,faq_15,Regional data residency is available in EU and...,0.000000


## 8. Evaluate retrieval quality


In [14]:
def reciprocal_rank(retrieved_ids, expected_id):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id == expected_id:
            return 1.0 / rank
    return 0.0


rows = []
for _, row in queries.iterrows():
    sem_ids = semantic_search(row["query"], k=3)["doc_id"].tolist()
    key_ids = keyword_search(row["query"], k=3)["doc_id"].tolist()

    rows.append(
        {
            "query": row["query"],
            "expected_doc_id": row["expected_doc_id"],
            "semantic_hit@3": int(row["expected_doc_id"] in sem_ids),
            "semantic_mrr@3": reciprocal_rank(sem_ids, row["expected_doc_id"]),
            "keyword_hit@3": int(row["expected_doc_id"] in key_ids),
            "keyword_mrr@3": reciprocal_rank(key_ids, row["expected_doc_id"]),
        }
    )

eval_df = pd.DataFrame(rows)
eval_df.head()

,query,expected_doc_id,semantic_hit@3,semantic_mrr@3,keyword_hit@3,keyword_mrr@3
0,How do I recover my account password?,faq_01,1,1.0,1,1.0
1,When are monthly bills created?,faq_02,1,1.0,0,0.0
2,Can I change delivery address after ordering?,faq_03,1,1.0,1,1.0
3,Where can I activate MFA?,faq_04,1,1.0,0,0.0
4,How long do refund checks take?,faq_05,1,1.0,1,1.0


In [15]:
summary = pd.DataFrame(
    {
        "method": ["semantic_embedding_faiss", "keyword_tfidf"],
        "recall@3": [eval_df["semantic_hit@3"].mean(), eval_df["keyword_hit@3"].mean()],
        "mrr@3": [eval_df["semantic_mrr@3"].mean(), eval_df["keyword_mrr@3"].mean()],
    }
).round(3)
summary

,method,recall@3,mrr@3
0,semantic_embedding_faiss,1.000,0.967
1,keyword_tfidf,0.733,0.733


## 9. Production-style query helper


In [16]:
def search_knowledge_base(query: str, method: str = "semantic", k: int = 3):
    """Search the KB with the selected method (`semantic` or `keyword`)."""
    if method == "semantic":
        return semantic_search(query, k=k)
    if method == "keyword":
        return keyword_search(query, k=k)
    raise ValueError("method must be 'semantic' or 'keyword'")

search_knowledge_base("Where do I configure retention rules?", method="semantic", k=3)

,doc_id,text,score
0,faq_11,Data retention can be configured per project i...,0.563559
1,faq_10,Webhook retries use exponential backoff up to ...,0.250920
2,faq_08,Audit logs can be exported from the compliance...,0.229119


## 10. Summary

- Data is externalized under `data/07_data` for reproducibility.
- The notebook compares semantic and lexical retrieval with measurable metrics.
- This is a practical pattern for enterprise search and knowledge assistants.
